In [ ]:
# Load modules
import os
from openai import AzureOpenAI
from dotenv import load_dotenv

# Load .env
load_dotenv()


In [ ]:
# Replace the following with your set up
os.environ['API_ENDPOINT']='XX'
os.environ['API_DEPLOYMENT']='XX'
os.environ['API_VERSION']='XX'
os.environ['API_KEY']='XX' # keep in confidential

client = AzureOpenAI(azure_endpoint=os.getenv('API_ENDPOINT'), api_version=os.getenv('API_VERSION'),
                     api_key=os.getenv('API_KEY'))

In [ ]:
# Replace the following with your path
%cd "/path/to/your/folder"

In [ ]:
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import string

def preprocess_text(text):
    # Replace 'N/A' with an empty string
    text = text.replace('N/A', '')
    # Convert to lowercase
    text = text.lower()
    # Remove extra spaces
    text = ' '.join(text.split())
    return text

def cosine_distance(text1, text2):
    # Preprocess the texts
    text1 = preprocess_text(text1)
    text2 = preprocess_text(text2)
    
    # Create the CountVectorizer and transform the texts (remove punctuations but keep operators)
    vectorizer = CountVectorizer(token_pattern=r'\b\w+\b|[<>]').fit_transform([text1, text2])
    # Compute cosine similarity between the two texts
    similarity = cosine_similarity(vectorizer)[0][1]
    # Return cosine distance (1 - similarity)
    return 1 - similarity

def jaccard_distance(text1, text2):
    # Preprocess the texts
    text1 = preprocess_text(text1)
    text2 = preprocess_text(text2)
    
    # Create the CountVectorizer and transform the texts (remove punctuations but keep operators)
    vectorizer = CountVectorizer(token_pattern=r'\b\w+\b|[<>]').fit_transform([text1, text2])
    
    # Convert the sparse matrix to dense and get the binary occurrence of tokens
    binary_occurrences = vectorizer.toarray().astype(bool)
    
    # Calculate Jaccard distance
    intersection = np.logical_and(binary_occurrences[0], binary_occurrences[1]).sum()
    union = np.logical_or(binary_occurrences[0], binary_occurrences[1]).sum()
    
    return 1 - (intersection / union if union != 0 else 0)



In [ ]:
def process_nctid(nctid):
    import os
    from openai import AzureOpenAI
    from dotenv import load_dotenv
    import csv
    import numpy as np
    import json
    import pandas as pd
    from openpyxl import load_workbook, Workbook
    from openpyxl.styles import Font, Alignment
    from openpyxl.utils.dataframe import dataframe_to_rows
    import time  # Added for timing
    from datetime import datetime  # Added for timestamp

    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    from scipy.spatial.distance import jaccard
    from openpyxl.utils import get_column_letter 
    import sys


    # Create a dictionary to store prompt metrics
    prompt_metrics = {
        'NCT ID': nctid,
        'Timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'Input Tokens': 0,
        'Output Tokens': 0,
        'Processing Time (seconds)': 0,
        'Prompt Length': len(prompt) if 'prompt' in locals() else 0
    }

    ## Task 3: Match outcomes ##

    # Prepare dataset for Task 3 matching

    # 3.1 Generate a dataset with original texts and distance measures (prospective)

    # 3.1.1 Extract original texts

    out_link=f"out_pro/{nctid}_merged.json"

    with open(out_link, 'r') as file:
        data = json.load(file)

    df1 = pd.DataFrame({'Original text (prospective)': data})

    # Create identifier
    def extract_priority_and_number(data):
        identifier = data.get('Identifier', '')
        return identifier.strip()

    # Apply the function to create new column
    df1['Identifier (prospective)'] = df1['Original text (prospective)'].apply(extract_priority_and_number).str.capitalize()

    # 3.1.2 Generate distance measures 

    # Open and read the JSON file

    with open(f"out_pro/{nctid}_merged.json", 'r') as file:
        json_data1 = file.read()

    with open(f"out_last_cut/{nctid}_merged.json", 'r') as file:
        json_data2 = file.read()

    data1 = json.loads(json_data1)
    data2 = json.loads(json_data2)

    # remove field names and identifiers
    processed_data1 = []
    identifiers = []
    for item in data1:
        identifiers.append(item.pop('Identifier', 'Unknown'))
        processed_data1.append(' '.join(str(value) for value in item.values()))

    processed_data2 = []
    identifiers = []
    for item in data2:
        identifiers.append(item.pop('Identifier', 'Unknown'))
        processed_data2.append(' '.join(str(value) for value in item.values()))


    # Generate distance measures
    # Initialize empty lists to store the results
    cosine_results = []
    jaccard_results = []

    # Loop through each element in processed_data1
    for i, text1 in enumerate(processed_data1):
        # Initialize lists to store the distances for the current i
        cosine_row = []
        jaccard_row = []

        # Loop through each element in processed_data2
        for j, text2 in enumerate(processed_data2):
            # Compute distances
            cosine_dist = cosine_distance(text1, text2)
            jaccard_dist = jaccard_distance(text1, text2)

            # Append distances to the current row lists
            cosine_row.append(cosine_dist)
            jaccard_row.append(jaccard_dist)

        # Append the current row lists to the results lists
        cosine_results.append(cosine_row)
        jaccard_results.append(jaccard_row)

    # The results are now stored in nested lists where each sublist represents
    # the distances for one element in processed_data1 against all elements in processed_data2
    # Create a DataFrame with the results
    dist1 = pd.DataFrame({
        'cosine_distance': cosine_results,
        'jaccard_distance': jaccard_results
    })


    # 3.1.3 Add distances measure
    # Append the column
    table1 = pd.concat([df1, dist1], axis=1)

    # 3.1.4 Save to a string variable
    table1_string = table1.to_json(orient='records')

    
    # 3.2 Generate a dataset with original texts and distance measures (last)

    # 3.2.1 Extract original texts

    out_link=f"out_last_cut/{nctid}_merged.json"

    with open(out_link, 'r') as file:
        data = json.load(file)

    df2 = pd.DataFrame({'Original text (last)': data})

    # Add the Identifier column
    df2['Identifier (last)'] = df2['Original text (last)'].apply(extract_priority_and_number).str.capitalize()

    # 3.2.2 Save to a string variable
    table2_string = df2.to_json(orient='records')
    ##print(table2_string)

    # 3.3 Match the outcomes
    # 3.3.1 Construct prompt
    with open("Prompts/Matching/Task3_mat.txt", 'r') as file:
        prompt1 = file.read()

    prompt2="# Input"
    prompt3="Here is the table of outcomes in the prospective version from ClinicalTrials.gov:"
    prompt4=table1_string
    prompt5="Here is the table of outcomes in the last version from ClinicalTrials.gov: "
    prompt6=table2_string

    prompt = prompt1 + "\n" + prompt2+ "\n" + prompt3 + "\n" + prompt4 + "\n" + prompt5 + "\n" + prompt6 + "\n" + prompt1


    # 3.3.2 Run prompt
    message = [{'role': 'user', 'content': prompt}]
    start_time = time.time()  # Start timing
    response3 = client.chat.completions.create(model=os.getenv('API_DEPLOYMENT'), 
                                               messages=message, 
                                               seed=66,
                                               max_tokens=4096,
                                               temperature=0,
                                               top_p=1)
    end_time = time.time()  # End timing
    elapsed_time = end_time - start_time
    
    # Store metrics
    prompt_metrics['Input Tokens'] = response3.usage.prompt_tokens
    prompt_metrics['Output Tokens'] = response3.usage.completion_tokens
    prompt_metrics['Processing Time (seconds)'] = elapsed_time
    
    print(f"API processing time for {nctid}: {elapsed_time:.2f} seconds")
    print(f"Tokens used - Input: {response3.usage.prompt_tokens}, Output: {response3.usage.completion_tokens}")

    # 3.3.3 Save results to a local file

    # Extract the content part from the OpenAIObject
    content3 = response3.choices[0].message.content

    # Parse the content string as JSON
    table_data3 = json.loads(content3.strip("```json"))

    # Extract the table columns and rows
    columns = table_data3["table"]["columns"]
    rows = table_data3["table"]["rows"]

    # Create a DataFrame from the rows and columns
    output3 = pd.DataFrame(rows, columns=columns)
    
    # Write the table data to a CSV file
    output3.to_csv(f"{nctid}_table3.csv", index=False, encoding='utf-8-sig')

    # 3.3.4 Save prompt metrics to Excel file
    def save_metrics_to_excel(metrics):
        excel_file = "prompt_metrics.xlsx"
        
        # Convert metrics to DataFrame
        metrics_df = pd.DataFrame([metrics])
        
        try:
            # Try to load existing file
            book = load_workbook(excel_file)
            
            # Get existing sheet or create new
            if 'Prompt Metrics' in book.sheetnames:
                startrow = book['Prompt Metrics'].max_row
                # Append to existing sheet
                with pd.ExcelWriter(excel_file, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
                    metrics_df.to_excel(writer, sheet_name='Prompt Metrics', 
                                        startrow=startrow,
                                        header=False,
                                        index=False)

            else:
                # Create new sheet in existing file
                with pd.ExcelWriter(excel_file, engine='openpyxl', mode='a') as writer:
                    metrics_df.to_excel(writer, sheet_name='Prompt Metrics', index=False)
                    
        except FileNotFoundError:
            # Create new file if doesn't exist
            metrics_df.to_excel(excel_file, sheet_name='Prompt Metrics', index=False)

    # [Rest of your existing code]
    save_metrics_to_excel(prompt_metrics)


In [ ]:
import time

# Read NCTIDs from a text file
with open('nctid_new150.txt', 'r') as file:
    nctids = [line.strip() for line in file if line.strip()]

# Process each NCTID
# Initialize a counter variable
n = 1

for nctid in nctids:
    print(f"Processing NCTID {n}/{len(nctids)}: {nctid}")
    process_nctid(nctid)
    print(f"Finished processing NCTID: {nctid}")
    
    # Optional: Add a delay between processing each NCTID to avoid rate limiting
    time.sleep(5)
    
    # Increment the counter
    n += 1

print("All NCTIDs processed.")


In [ ]:
import sys

# Stop the script
sys.exit()